# Stage 5 - Generate

## Data
Chroma DB

In [1]:
'''
1.1 Vectors DB Client
'''
import chromadb

client = chromadb.PersistentClient(path="chroma_kb") 
col = client.get_collection(name="knowledge_base")         

In [7]:
'''
1.2 Retrieve - dict
'''
from sentence_transformers import SentenceTransformer

THRESHOLD = 0.8
K = 5

embedder = SentenceTransformer("intfloat/multilingual-e5-base", device="mps")

def retrieve_dict(query, col=col, embedder=embedder, k=K, threshold=THRESHOLD):
  """Retrieval-пайплайн: embed query -> search -> filter -> return."""
  q_vec = embedder.encode("query: " + query, normalize_embeddings=True)
  res = col.query(query_embeddings=[q_vec.tolist()], n_results=k)
  out = []
  for doc, m, dist in zip(res["documents"][0],
                          res["metadatas"][0],
                          res["distances"][0]):
    sim = 1.0 - dist                        # Косинусное сходство
    if sim >= threshold:                    # Порог по сходству
      out.append({
        'id': m['id'],
        'doc_id': m['doc_id'], 
        'similarity': round(sim, 4), 
        'text': doc, 
      }
    )
  return out

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6678.56it/s]


In [ ]:
'''
1.3 load queries
'''
import json

with open("1_queries.json", encoding="utf-8") as f:
  queries = json.load(f)

In [ ]:
'''
2. LLM Generate
'''
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8080/v1", api_key="not-needed")
LLM = "qwen3.5"                                       # Имя сервер игнорирует
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}


def generate(messages, max_tokens=160):
    """Один вызов локальной LLM. temperature=0 — для воспроизводимости,
    режим рассуждения Qwen3.5 отключён, поэтому сразу приходит готовый текст."""
    resp = client.chat.completions.create(
        model=LLM, messages=messages, temperature=0.0,
        max_tokens=max_tokens, extra_body=NO_THINK)
    return resp.choices[0].message.content.strip()

In [15]:
SYSTEM = ("Ты — ассистент поддержки. Отвечай ТОЛЬКО по приведённым фрагментам "
          "базы знаний. Если ответа в них нет, ответь: «В базе знаний ответа "
          "нет». Отвечай кратко, одним-двумя предложениями.")


def build_messages(query, chunk_dicts):
    """Собирает промпт: system-инструкция + нумерованный контекст + вопрос."""
    # print('>>> chunk_dicts: ', chunk_dicts)
    context = "\n".join(f"[{n}] {i['text']}" for n, i in enumerate(chunk_dicts, 1))
    user = f"Фрагменты базы знаний:\n{context}\n\nВопрос: {query}"
    return [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": user}]

def rag_answer(query, search_fn, k=5):
    """Полный RAG: retrieval -> промпт -> генерация."""
    ret = search_fn(query, k=5)
    msg = build_messages(query, ret)
    answer = generate(msg)
    return answer, [i['id'] for i in ret], [i['text'] for i in ret]

In [16]:
print('>>> exact queries')
for q in [x for x in queries if x["kind"] == "exact"][:3]:
  answer, ids, texts = rag_answer(q["query"], retrieve_dict)
  print(f"\nВ: {q['query']}")
  print(f"   чанки: {ids}  ({[i[:34] for i in texts]})")
  print(f"   О: {answer}")

print('\n\n>>> semantic queries')
for q in [x for x in queries if x["kind"] == "semantic"][:3]:
  answer, ids, texts = rag_answer(q["query"], retrieve_dict)
  print(f"\nВ: {q['query']}")
  print(f"   чанки: {ids}  ({[i[:34] for i in texts]})")
  print(f"   О: {answer}")

print('\n\n>>> multi queries')
for q in [x for x in queries if x["kind"] == "multi"][:3]:
  answer, ids, texts = rag_answer(q["query"], retrieve_dict)
  print(f"\nВ: {q['query']}")
  print(f"   чанки: {ids}  ({[i[:34] for i in texts]})")
  print(f"   О: {answer}")

print('\n\n>>> gaps')
for q in [x for x in queries if x["kind"] == "gap"][:3]:
  answer, ids, texts = rag_answer(q["query"], retrieve_dict)
  print(f"\nВ: {q['query']}")
  print(f"   чанки: {ids}  ({[i[:34] for i in texts]})")
  print(f"   О: {answer}")

>>> exact queries

В: что означает ошибка P-102
   чанки: [98, 207, 159, 160, 150]  (['# Ошибка P-102: превышен лимит про', 'Ошибка P-109 появляется, когда на ', 'Ошибка P-106 появляется, когда пле', 'Ошибка P-104 появляется, когда уст', 'Ошибка P-105 появляется, когда на '])
   О: Ошибка P-102 появляется, когда число одновременных просмотров на аккаунте превышает лимит тарифа. Для решения проблемы нужно проверить количество активных экранов, обновить приложение и при необходимости перейти на тариф с большим лимитом.

В: какая версия webOS или Tizen нужна для приложения на телевизоре
   чанки: [171, 99, 150, 199, 161]  (['# На каких устройствах работает «Ф', '# Поддерживаемые устройства На как', 'Ошибка P-105 появляется, когда на ', 'Приложение на телевизоре обновляет', 'Если вы не знаете, к какой платфор'])
   О: Для работы приложения на телевизорах Samsung требуется Tizen с моделями с 2018 года, а на LG — webOS 4.0 и новее.

В: обязателен ли HDCP 2.2 для просмотра в 4K
   чанки: [128